In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

In [16]:
def build_resilient_chain(
    primary_model_name: str = "ollama:llama3.2",
    backup_model_name: str = "ollama:llama3.2.1b",
    timeout_seconds: float = 8.0,
    max_retries: int = 2,
):
    prompt = ChatPromptTemplate.from_template(
        "Summarize this support issue in 2 short bullet point in Hinglish:\n{issue}"
    )

    primary_model = init_chat_model(
    primary_model_name,
    temperature=0,
    timeout=timeout_seconds,
    max_retries=max_retries,
  )

    backup_model = init_chat_model(
        backup_model_name,
        temperature=0,
        timeout=timeout_seconds,
        max_retries=max_retries,
    )

    primary_chain = prompt | primary_model
    backup_chain = prompt | backup_model

    return primary_chain.with_fallbacks([backup_chain])

In [23]:
def ask_resilient(issue: str, chain) -> str:
    try:
        chunks = []
        for chunk in chain.stream({"issue": issue}):
            chunks.append(chunk.content)
        return "".join(chunks)
    except Exception as exc:
        print("Actual Error:", exc)
        return (
            "Abhi Hamari Ai service temprary issue ya slow respose ki bajah se "
            "unavailable hai. Please try again after some minutes."
        )

In [24]:
if __name__ == "__main__":
    chain = build_resilient_chain()
    print("Normal test:\n", ask_resilient("Payment kat gaya par order confirm nhi hua.", chain))

    wrong_model_chain = build_resilient_chain(
        primary_model_name="ollama:xyz",
        backup_model_name="ollama:llama3.2",
    )
    print("\nWrong model test:\n", ask_resilient("App login error a rha hai", wrong_model_chain))

    short_timeout_chain = build_resilient_chain(timeout_seconds=0.001)
    print("\nShort timeout test:\n", ask_resilient("Invoice Download nhi ho rha hai", short_timeout_chain))
          


Normal test:
 Here is a summary of the support issue in 2 short bullet points in Hinglish:

• Payment failed: Order not confirmed.
• Need help with payment issue.

Wrong model test:
 Here is a summary of the support issue in 2 short bullet points in Hinglish:

• App login error ho raha hai.
• Kya aapko help chahiye?

Short timeout test:
 Here is a summary of the support issue in 2 short bullet points in Hinglish:

• Invoice download nahi ho raha hai.
• Kya aapko koi aur madad ki zarurat hai?
